[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/09_alumno_ensamble.ipynb)

# MLY1101 · Machine Learning — Actividad 3.2
## Modelos de ensamble

**Resultado de aprendizaje (RA3):** elabora soluciones avanzadas de aprendizaje automático
mediante la optimización de hiperparámetros, técnicas de ensamble y validación cruzada, para
garantizar la precisión y generalización del modelo frente a objetivos de negocio complejos.

**Indicador de logro (IL 3.2):** desarrolla modelos basados en técnicas de ensamble para mitigar
problemas de sesgo y varianza en escenarios de negocio complejos.

---

### De dónde viene la pregunta

La Actividad 3.1 terminó con un resultado incómodo: ajustar hiperparámetros **no mejoró nada**.
La reacción natural es *"entonces probemos un modelo más potente"*.

Eso es exactamente lo que hacemos hoy. Y también lo vamos a medir.

---

### Sesgo y varianza, en una tabla

El error de un modelo se descompone en dos partes que se combaten de forma distinta:

| | **Sesgo** | **Varianza** |
|---|---|---|
| Qué es | El modelo es demasiado simple para el problema | El modelo cambia mucho según los datos que le tocaron |
| Cómo se ve | Falla igual en entrenamiento y en prueba | Va perfecto en entrenamiento y mal en prueba |
| Ejemplo | Una recta para separar algo curvo | Un árbol sin límite de profundidad |
| Cómo se reduce | Modelo más flexible, mejores variables | **Promediar modelos**, más datos, regularizar |

**Los ensambles atacan sobre todo la varianza.** Promediar modelos que se equivocan en cosas
distintas cancela parte del error.

**Y no arreglan el sesgo.** Si todos los modelos comparten el mismo punto ciego —porque las
variables no contienen la información que hace falta—, promediarlos no lo elimina. Diez modelos
mirando por la misma ventana no ven más.

---

### Las tres familias

| Familia | Idea | Ejemplo |
|---|---|---|
| **Bagging** | Entrenar en paralelo sobre muestras distintas y promediar | Bosque aleatorio |
| **Boosting** | Entrenar en serie: cada modelo corrige los errores del anterior | Gradient boosting |
| **Votación / apilamiento** | Combinar modelos **distintos entre sí** | `VotingClassifier` |

> **Detalle que suele pasar desapercibido:** el bosque aleatorio que llevas usando desde la
> Actividad 2.2 **ya es un ensamble**. Son 200 árboles votando. No vamos a introducir los
> ensambles: llevamos toda la asignatura usando uno.

---

### Al final de la sesión debes entregar

La comparación de modelos con **métrica y costo juntos**, y una respuesta argumentada a: *¿el
ensamble más complejo gana lo suficiente para justificar lo que cuesta?*

---
## Preparación del entorno

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO.resolve()
else:
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ / "kedro_mly1101" / "src"))

RUTA_DATOS = RAIZ / "datos" / "crudos" / "detecciones_waymo_like.csv"
RUTA_PARAMETROS = RAIZ / "kedro_mly1101" / "conf" / "base" / "parameters.yml"
print("Colab:", EN_COLAB, "| dataset:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

from kedro_mly1101.pipelines.preprocesamiento import nodes as limpieza
from kedro_mly1101.pipelines.supervisado import nodes as supervisado
from kedro_mly1101.pipelines.optimizacion import nodes as optimizacion

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")

PARAMETROS = yaml.safe_load(RUTA_PARAMETROS.read_text(encoding="utf-8"))
CONFIG, FUGA, AJUSTE = PARAMETROS["modelo"], PARAMETROS["fuga"], PARAMETROS["ajuste"]

# La misma cadena de siempre: limpieza del RA1 -> partición del RA2.
crudo = pd.read_csv(RUTA_DATOS)
paso = limpieza.normalizar_categorias(crudo, PARAMETROS["mapas_categorias"])
paso = limpieza.descubrir_faltantes(paso, PARAMETROS["centinelas"])
paso = limpieza.marcar_imposibles(paso, PARAMETROS["reglas_dominio"])
limpio = limpieza.quitar_duplicados_y_constantes(paso, PARAMETROS["columnas_a_descartar"])

marcada = supervisado.particionar(
    supervisado.preparar_variables(limpio, CONFIG, FUGA), CONFIG
)
entrena = marcada[marcada["particion"] == "entrenamiento"]

print(f"Entrenamiento: {len(entrena):,} filas en {entrena[CONFIG['grupo']].nunique()} segmentos")
print(f"Métrica de trabajo: {AJUSTE['metrica']}  ·  pliegues: {AJUSTE['n_pliegues']}")

---
# Bloque 1 · ⭐ Contra qué se compara

Un resultado suelto no significa nada. Antes de comparar modelos complejos hace falta el piso:

| Candidato | Por qué está en la lista |
|---|---|
| **Baseline** | Responde siempre la clase mayoritaria. Si tu modelo no le gana, no hay modelo |
| **Árbol de decisión** | El más simple que aprende algo. Interpretable: se puede dibujar |
| **Regresión logística** | El modelo lineal. Si gana, el problema era lineal y sobraba lo demás |
| **Bosque aleatorio** | Bagging. El que venimos usando |
| **Gradient boosting** | Boosting. Corrige errores en serie |
| **Votación** | Combina árbol + bosque + boosting |

### ✏️ TODO 1 — Ejecutar la comparación

`optimizacion.comparar_ensambles()` evalúa los seis con validación cruzada **por grupo** y
cronometra cada uno.

*(Tarda unos 15 segundos: son 30 entrenamientos.)*

In [ ]:
# TODO 1: compara los seis candidatos con validación cruzada por grupo.
comparacion = optimizacion.____(marcada, CONFIG, AJUSTE)
comparacion

### ✏️ TODO 2 — Lo primero que hay que mirar

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

1. ¿Cuánto le saca la **regresión logística** al baseline?
2. ¿Qué te dice eso sobre la naturaleza del problema?

---
# Bloque 2 · Métrica y costo, juntos

Fíjate en que la tabla trae una columna `segundos`. No es decoración.

La pregunta de esta actividad **no** es *"¿cuál saca el número más alto?"*. Es:

> **¿Gana lo suficiente para justificar lo que cuesta?**

Un modelo que gana 0,004 y tarda cuatro veces más no es mejor: es más caro.

### ✏️ TODO 3 — El costo relativo

In [ ]:
# TODO 3: ¿cuánto cuesta cada punto de mejora?
costo = comparacion.copy()
costo["veces_mas_lento"] = (costo["segundos"] / costo["segundos"].____()).round(1)
costo["ganancia_vs_arbol"] = (
    costo["media"] - costo.loc[costo["modelo"] == "arbol", "media"].iloc[0]
).round(4)
costo[["modelo", "media", "ganancia_vs_arbol", "segundos", "veces_mas_lento"]]

---
# Bloque 3 · ⭐⭐ El ensamble por votación

Ya tenemos tres modelos buenos. La intuición dice que combinarlos debería dar algo mejor que
cualquiera de los tres: cada uno se equivoca en cosas distintas y el voto cancela errores.

### ✏️ TODO 4 — Antes de mirar la tabla, apuesta

**Creo que el ensamble por votación quedará:** `____`
*(por encima del bosque / igual / por debajo)*

### ✏️ TODO 5 — La comparación directa

In [ ]:
# TODO 5: ¿el ensamble le gana al mejor modelo individual?
duelo = comparacion[comparacion["modelo"].isin(["bosque_aleatorio", "ensamble_votacion"])]
print(duelo.to_string(index=False), "\n")

mejor_solo = duelo[duelo["modelo"] == "bosque_aleatorio"].iloc[0]
ensamble = duelo[duelo["modelo"] == "____"].iloc[0]

diferencia = ensamble["media"] - mejor_solo["media"]
sobrecosto = ensamble["segundos"] / mejor_solo["segundos"]

print(f"Diferencia en F1-macro : {diferencia:+.4f}")
print(f"Ruido entre pliegues   : {mejor_solo['desv_entre_pliegues']:.4f}")
print(f"Sobrecosto en tiempo   : {sobrecosto:.2f}×")

In [ ]:
# Autochequeo
assert diferencia < 0, "revisa: el ensamble debería quedar POR DEBAJO del bosque solo"
assert abs(diferencia) < mejor_solo["desv_entre_pliegues"], (
    "y la diferencia debería ser menor que el ruido entre pliegues"
)
print("✅ El ensamble quedó por debajo del bosque, y la diferencia es menor")
print("   que el ruido: no hay evidencia de que ninguno sea mejor.")
print(f"   Pero el ensamble tarda {sobrecosto:.0%} de lo que tarda el bosque.")
print()
print("   Mismo desempeño demostrable, más costo, menos interpretable.")

### ✏️ TODO 6 — Por qué no funcionó

**✍️ Tu respuesta:**

*(doble clic aquí y escribe)*

El ensamble combina árbol, bosque y boosting. No mejoró. Da una explicación, sabiendo que
**los ensambles reducen varianza, no sesgo**.

*Pista: mira qué tienen en común los tres modelos combinados.*

---
# Bloque 4 · ¿Sesgo o varianza?

Si ni el ajuste ni el ensamble mejoran, la pregunta es **qué limita** al modelo.

| Síntoma | Diagnóstico | Qué hacer |
|---|---|---|
| Va mucho mejor en entrenamiento que en validación | **Varianza** | Más datos, regularizar, promediar |
| Va parecido en ambos, y ambos mediocres | **Sesgo** | Mejores variables, modelo más flexible |

### ✏️ TODO 7 — El diagnóstico

In [ ]:
# TODO 7: ¿el modelo sufre de sesgo o de varianza?
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupKFold, cross_val_score

X = entrena[CONFIG["variables"]]
y = entrena[CONFIG["objetivo"]]
grupos = entrena[CONFIG["grupo"]]

modelo = RandomForestClassifier(
    n_estimators=200, max_depth=12, class_weight="balanced",
    random_state=CONFIG["semilla"], n_jobs=-1,
)
modelo.fit(X, y)

en_entrenamiento = f1_score(y, modelo.____(X), average="macro")
en_validacion = ____(
    modelo, X, y, groups=grupos,
    cv=GroupKFold(n_splits=AJUSTE["n_pliegues"]), scoring=AJUSTE["metrica"], n_jobs=-1,
).mean()

print(f"F1-macro en entrenamiento : {en_entrenamiento:.4f}")
print(f"F1-macro en validación    : {en_validacion:.4f}")
print(f"Brecha                    : {en_entrenamiento - en_validacion:.4f}")

**✍️ Tu respuesta al TODO 7:**

*(doble clic aquí y escribe)*

1. ¿El problema es sesgo o varianza?
2. Según ese diagnóstico, ¿qué habría que hacer para mejorar de verdad?
3. ¿Por qué eso explica que el ajuste y el ensamble no sirvieran?

---
# Cierre · Informe de ensamble

### Los candidatos

| Modelo | Familia | F1-macro | Desv. | Segundos |
|---|---|---|---|---|
| `____` | | | | |
| `____` | | | | |
| `____` | | | | |

**Baseline:** `____` · **Mejor modelo individual:** `____` · **Ensamble:** `____`

### La comparación que importa

**Diferencia entre el ensamble y el mejor individual:** `____`
**Ruido entre pliegues:** `____`
**¿Es distinguible?** `____`
**Sobrecosto en tiempo:** `____`

**Decisión y por qué:** `____`

> Si eliges el más simple, **dilo con el argumento del costo y la interpretabilidad**, no como
> si te conformaras. Elegir el modelo suficiente es una decisión de ingeniería, no una renuncia.

### Diagnóstico

**¿Sesgo o varianza?** `____` · **Evidencia:** `____`
**Qué haría falta para mejorar de verdad:** `____`